In [ ]:
# Mount Google Drive (optional, if you're working with Google Colab)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Install necessary libraries
!pip install deepface scikit-learn opencv-python-headless tqdm mtcnn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 6.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.6 MB/s eta 0:00:00
  Created wheel for fire: filename=fire-0.7.0-py3-none-any.whl size=114249 sha256=232059c40168cc5cd1870560ac6570aa3784ca78c1d2e446d2c92f28be1a31b7
  Stored in directory: /root/.cache/pip/wheels/19/39/2f/2d3cadc408a8804103f1c34ddd4b9f6a93497b11fa96fe738e
Successfully built fire


In [ ]:
import os
import cv2
import numpy as np
from deepface import DeepFace
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from tqdm import tqdm
import random
from mtcnn import MTCNN

24-11-06 23:09:25 - Directory /root/.deepface has been created
24-11-06 23:09:25 - Directory /root/.deepface/weights has been created


In [ ]:
# Initialize MTCNN detector
mtcnn_detector = MTCNN()

In [ ]:
# Function to ensure folder exists
def ensure_folder_exists(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

In [ ]:
# Function for image augmentation with additional variations
def augment_image(image):
    augmented_images = []

    # Rotation
    for angle in [10, -10, 20, -20, 30, -30]:
        M = cv2.getRotationMatrix2D((image.shape[1] // 2, image.shape[0] // 2), angle, 1.0)
        rotated = cv2.warpAffine(image, M, (image.shape[1], image.shape[0]))
        augmented_images.append(rotated)

    # Horizontal flip
    flipped = cv2.flip(image, 1)
    augmented_images.append(flipped)

    # Zoom
    zoom_factor = random.choice([1.1, 1.2])
    centerX, centerY = image.shape[1] // 2, image.shape[0] // 2
    radiusX, radiusY = int(centerX / zoom_factor), int(centerY / zoom_factor)
    cropped_zoomed = image[centerY - radiusY:centerY + radiusY, centerX - radiusX:centerX + radiusX]
    cropped_zoomed = cv2.resize(cropped_zoomed, (image.shape[1], image.shape[0]))
    augmented_images.append(cropped_zoomed)

    # Brightness and contrast adjustments
    for alpha, beta in [(1.2, 30), (0.8, -30)]:
        adjusted = cv2.convertScaleAbs(image, alpha=alpha, beta=beta)
        augmented_images.append(adjusted)

    # Add noise
    noise = np.random.normal(0, 25, image.shape).astype(np.uint8)
    noisy_image = cv2.add(image, noise)
    augmented_images.append(noisy_image)

    return augmented_images

In [ ]:
# Face detection function using MTCNN with confidence threshold
def detect_and_crop_face(image, confidence_threshold=0.3):
    # Detect faces in the image using MTCNN
    detections = mtcnn_detector.detect_faces(image)
    for detection in detections:
        confidence = detection['confidence']
        if confidence > confidence_threshold:
            x, y, width, height = detection['box']
            face = image[y:y+height, x:x+width]
            return face
    return None

In [ ]:
# Image processing function
def preprocess_image(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    equalized = cv2.equalizeHist(gray)
    smoothed = cv2.medianBlur(equalized, 5)
    sharpen_kernel = np.array([[-1, -1, -1], [-1, 9, -1], [-1, -1, -1]])
    sharpened = cv2.filter2D(smoothed, -1, sharpen_kernel)
    alpha, beta = 1.2, 10
    adjusted = cv2.convertScaleAbs(sharpened, alpha=alpha, beta=beta)
    preprocessed_image = cv2.merge([adjusted, adjusted, adjusted])
    return preprocessed_image

In [ ]:
# Function to create a collage of two images side by side
def create_collage(image1, image2, save_path):
    image1 = cv2.resize(image1, (image2.shape[1], image2.shape[0]))
    image1 = image1.astype(image2.dtype)

    collage = cv2.hconcat([image1, image2])
    save_path = os.path.splitext(save_path)[0] + ".jpg"
    cv2.imwrite(save_path, collage)

In [ ]:
# Function to save original and processed image collage for each dataset image
def save_processed_collages(train_dir, output_collage_dir):
    ensure_folder_exists(output_collage_dir)

    for person_name in os.listdir(train_dir):
        person_folder = os.path.join(train_dir, person_name)
        if not os.path.isdir(person_folder):
            continue

        person_collage_dir = os.path.join(output_collage_dir, person_name)
        ensure_folder_exists(person_collage_dir)

        for image_name in os.listdir(person_folder):
            image_path = os.path.join(person_folder, image_name)
            image = cv2.imread(image_path)
            face = detect_and_crop_face(image)
            if face is None:
                continue
            processed_image = preprocess_image(face)

            # Create collage between original and processed images
            original_resized = cv2.resize(face, (processed_image.shape[1], processed_image.shape[0]))
            collage = cv2.hconcat([original_resized, processed_image])
            collage_save_path = os.path.join(person_collage_dir, f"collage_{image_name}")

            # Ensure the collage is saved as .jpg
            collage_save_path = os.path.splitext(collage_save_path)[0] + ".jpg"
            cv2.imwrite(collage_save_path, collage)

In [ ]:
# Function to build embeddings with additional augmentation
def build_known_embeddings(train_dir):
    embeddings = []
    labels = []
    for person_name in os.listdir(train_dir):
        person_folder = os.path.join(train_dir, person_name)
        if not os.path.isdir(person_folder):
            continue
        for image_name in os.listdir(person_folder):
            image_path = os.path.join(person_folder, image_name)
            image = cv2.imread(image_path)
            face = detect_and_crop_face(image)
            if face is None:
                continue
            processed_image = preprocess_image(face)
            try:
                embedding = DeepFace.represent(processed_image, model_name='ArcFace', enforce_detection=False)[0]["embedding"]
                embeddings.append(embedding)
                labels.append(person_name)
                for aug_img in augment_image(processed_image):
                    augmented_embedding = DeepFace.represent(aug_img, model_name='ArcFace', enforce_detection=False)[0]["embedding"]
                    embeddings.append(augmented_embedding)
                    labels.append(person_name)
            except Exception:
                continue
    return np.array(embeddings), labels

In [ ]:
# Accuracy evaluation function with skipped folder tracking
def evaluate_accuracy_by_folder(val_dir, classifier, label_encoder, scaler, pca, train_dir, output_dir, tolerance=0.8):
    correct_folders = 0
    total_folders = 0
    skipped_folders = []  # Tracking skipped folders

    recognized_dir = os.path.join(output_dir, "recognized_faces")
    unrecognized_dir = os.path.join(output_dir, "unrecognized_faces")
    ensure_folder_exists(recognized_dir)
    ensure_folder_exists(unrecognized_dir)

    for person_name in os.listdir(val_dir):
        person_folder = os.path.join(val_dir, person_name)
        if not os.path.isdir(person_folder):
            continue

        total_folders += 1
        recognized_images = 0
        total_images = 0

        for image_name in os.listdir(person_folder):
            image_path = os.path.join(person_folder, image_name)
            image = cv2.imread(image_path)
            face = detect_and_crop_face(image)
            if face is None:
                continue

            processed_image = preprocess_image(face)
            try:
                embedding = DeepFace.represent(processed_image, model_name='ArcFace', enforce_detection=False)[0]["embedding"]
                embedding = scaler.transform([embedding])
                embedding = pca.transform(embedding)
                predicted_label = classifier.predict(embedding)[0]
                predicted_name = label_encoder.inverse_transform([predicted_label])[0]

                if predicted_name == person_name:
                    recognized_images += 1
                total_images += 1
            except Exception:
                continue

        if total_images == 0:
            # Jika tidak ada gambar yang diproses dalam folder ini, catat sebagai terlewat
            skipped_folders.append(person_name)
            continue

        recognition_rate = recognized_images / total_images if total_images > 0 else 0
        if recognition_rate >= tolerance:
            correct_folders += 1
            person_recognized_dir = os.path.join(recognized_dir, person_name)
            ensure_folder_exists(person_recognized_dir)

            # Memuat gambar asli dari folder latih
            train_image_path = os.path.join(train_dir, person_name, os.listdir(os.path.join(train_dir, person_name))[0])
            train_image = cv2.imread(train_image_path)

            # Membuat kolase dengan gambar dari folder validasi
            for image_name in os.listdir(person_folder):
                val_image = cv2.imread(os.path.join(person_folder, image_name))
                collage_save_path = os.path.join(person_recognized_dir, f"collage_{image_name}")
                create_collage(train_image, val_image, collage_save_path)
        else:
            person_unrecognized_dir = os.path.join(unrecognized_dir, person_name)
            ensure_folder_exists(person_unrecognized_dir)
            for image_name in os.listdir(person_folder):
                val_image = cv2.imread(os.path.join(person_folder, image_name))
                # Save as .jpg
                unrecognized_save_path = os.path.join(person_unrecognized_dir, os.path.splitext(image_name)[0] + ".jpg")
                cv2.imwrite(unrecognized_save_path, val_image)

    # Hitung akurasi
    accuracy = (correct_folders / (total_folders - len(skipped_folders))) * 100 if (total_folders - len(skipped_folders)) > 0 else 0
    print(f"Folder-based accuracy: {accuracy:.2f}%")
    print(f"\nNumber of recognized folders: {correct_folders}")
    print(f"Number of unrecognized folders: {total_folders - correct_folders - len(skipped_folders)}")
    print(f"Number of skipped folders: {len(skipped_folders)}")

    # Tampilkan folder yang terlewat
    if skipped_folders:
        print("\nSkipped folders (not processed due to missing images or errors):")
        for folder in skipped_folders:
            print(folder)

    return accuracy

In [ ]:
# Setup paths for dataset and output
train_dir = '/content/drive/MyDrive/Colab Notebooks/nist_2/train'
val_dir = '/content/drive/MyDrive/Colab Notebooks/nist_2/val'
output_dir = '/content/drive/MyDrive/Colab Notebooks/comvisfixmtcnn/outputfaces_comvisfixmtcnn'

In [ ]:
# Build embeddings with augmentation
train_embeddings, train_labels = build_known_embeddings(train_dir)
label_encoder = LabelEncoder()
train_labels = label_encoder.fit_transform(train_labels)
scaler = StandardScaler()
train_embeddings = scaler.fit_transform(train_embeddings)
pca = PCA(n_components=50)
train_embeddings = pca.fit_transform(train_embeddings)

24-11-06 23:09:41 - arcface_weights.h5 will be downloaded to /root/.deepface/weights/arcface_weights.h5


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/arcface_weights.h5
To: /root/.deepface/weights/arcface_weights.h5
100%|██████████| 137M/137M [00:01<00:00, 113MB/s]


In [ ]:
# Train KNN
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(train_embeddings, train_labels)

KNeighborsClassifier()

In [ ]:
# Evaluate with tolerance
print("\nEvaluating KNN model:")
evaluate_accuracy_by_folder(val_dir, knn_model, label_encoder, scaler, pca, train_dir, output_dir, tolerance=0.8)


Evaluating KNN model:
Folder-based accuracy: 81.63%

Number of recognized folders: 120
Number of unrecognized folders: 27
Number of skipped folders: 12

Skipped folders (not processed due to missing images or errors):
S027
S085
S124
S148
S132
S173
S198
S236
S254
S263
S291
S325


81.63265306122449

In [ ]:
# Save processed collages
output_collage_dir = '/content/drive/MyDrive/Colab Notebooks/comvisfixmtcnn/hasilcitra_comvisfixmtcnn'
save_processed_collages(train_dir, output_collage_dir)